<a href="https://colab.research.google.com/github/gauravd12345/miniCLIP/blob/main/miniCLIP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("eeshawn/flickr30k")

print("Path to dataset files:", path)

images = path + '/flickr30k_images'
captions = path + '/captions.txt'

100%|██████████| 4.08G/4.08G [00:50<00:00, 87.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/eeshawn/flickr30k/versions/1


In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

from tqdm import tqdm

from torchvision import transforms
from PIL import Image
from IPython.display import display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cuda


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.add_special_tokens({"bos_token": "<sos>", "eos_token": "<eos>", "pad_token": "<pad>"})

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

3

In [4]:
vocab_len = len(tokenizer)
chunk_size = 64      # max seq len for text transformer

batch_size = 64
epochs = 10
lr = 2.5e-4

""" Vision Transformer parameters """
H = 128               # image height
W = 128               # image width
C = 3                 # number of channels

P = 16                # patch resolution
N_p = (H * W) // P**2   # number of patches

""" Transformer parameters """
d_model = 512
d_k = 64
d_v = 64
h = 8
N = 6

""" CLIP parameters """
d_i = 512
d_t = 512
d_e = 512

In [5]:
import pandas as pd

df = pd.read_csv(captions)

image_names = df['image_name']
comment_numbers = df['comment_number']
comments = df['comment']

In [6]:
class ViTDataset(Dataset):
    def __init__(self, image_names, captions, transform=None):
        self.captions = captions
        self.image_names = image_names
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img = Image.open(f"{images}/{self.image_names[idx]}")
        if self.transform:
            img = self.transform(img)           # extract H, W, C values
        img = img.reshape(N_p, P**2 * C)        # flatten

        tokens = tokenizer(f"<sos> {self.captions[idx]} <eos>",
                           return_tensors="pt",
                           padding="max_length",
                           truncation=True,
                           max_length=chunk_size
                        )
        input_ids = tokens["input_ids"].squeeze(0)

        return img, input_ids


transform = transforms.Compose([ # resizing images
    transforms.Resize((H, W)),
    transforms.ToTensor()
])

dataset = ViTDataset(image_names, comments, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

for img, caption in dataloader:
    print(f"Samples per batch: {len(dataloader)}")
    print(f"Batched images shape: {img.shape}") # B, N, P**2 * C
    print(f"Batched captions shape: {caption.shape}")
    break


Samples per batch: 2484
Batched images shape: torch.Size([64, 64, 768])
Batched captions shape: torch.Size([64, 64])


In [7]:
import math

# training scheduler
total_steps = epochs * len(dataloader)
warmup_steps = int(0.01 * total_steps)

def lr_lambda(step): # cosine schedule
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

In [32]:
class Transformer(nn.Module):
  def __init__(self):
    super().__init__()

    self.embed = nn.Embedding(vocab_len, d_model)
    self.pos = nn.Embedding(chunk_size, d_model) # positional embedding

    self.W_q = nn.ModuleList([nn.ModuleList([nn.Linear(d_model, d_k) for _ in range(h)]) for _ in range(N)]) # q, k, v projections
    self.W_k = nn.ModuleList([nn.ModuleList([nn.Linear(d_model, d_k) for _ in range(h)]) for _ in range(N)])
    self.W_v = nn.ModuleList([nn.ModuleList([nn.Linear(d_model, d_v) for _ in range(h)]) for _ in range(N)])

    self.W_o = nn.ModuleList([nn.Linear(h * d_v, d_model) for _ in range(N)]) # final projection

    self.ffn1 = nn.ModuleList([nn.Linear(d_model, 4 * d_model) for _ in range(N)]) # ffn layer
    self.ffn2 = nn.ModuleList([nn.Linear(4 * d_model, d_model) for _ in range(N)])

    self.ln1 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(N)])
    self.ln2 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(N)])

    self.dropout = nn.Dropout(0.1)

    self.fc = nn.Linear(d_model, d_t // chunk_size) # final layer

  def multi_head_attention(self, x, layer_idx):
    W_tot = []
    for Q, K, V in zip(self.W_q[layer_idx], self.W_k[layer_idx], self.W_v[layer_idx]):
      Q_i = Q(x)
      K_i = K(x)
      V_i = V(x)

      alignment = torch.matmul(Q_i, K_i.transpose(-2, -1))                      # query-key alignment

      wei = self.dropout(torch.softmax(alignment / (d_k ** 0.5), dim=-1))       # alignment weights
      wei_value = torch.matmul(wei, V_i)                                        # weighted values

      W_tot.append(wei_value)

    out = self.W_o[layer_idx](torch.cat(W_tot, dim=2))
    return out

  def forward(self, x): # (batch_size, chunk_size)
    p = torch.arange(x.size(1)).to(x.device)
    x = self.dropout(self.embed(x) + self.pos(p))     # word & positional embedding
    for i in range(N):
      out = self.multi_head_attention(x, i)  # multi head attention
      out = self.ln1[i](out + x)             # layernorm + residual connection

      fn = self.ffn1[i](out)                 # ffn
      fn = torch.relu(fn)
      fn = self.dropout(self.ffn2[i](fn))

      out = self.ln2[i](fn + out)
      x = out

    out = self.fc(out).flatten(start_dim=1)
    return out


In [33]:
class VisionTransformer(nn.Module):
  def __init__(self):
    super().__init__()

    self.embed = nn.Linear(P**2 * C, d_model)
    self.pos = nn.Embedding(N_p, d_model) # positional embedding

    self.W_q = nn.ModuleList([nn.ModuleList([nn.Linear(d_model, d_k) for _ in range(h)]) for _ in range(N)]) # q, k, v projections
    self.W_k = nn.ModuleList([nn.ModuleList([nn.Linear(d_model, d_k) for _ in range(h)]) for _ in range(N)])
    self.W_v = nn.ModuleList([nn.ModuleList([nn.Linear(d_model, d_v) for _ in range(h)]) for _ in range(N)])

    self.W_o = nn.ModuleList([nn.Linear(h * d_v, d_model) for _ in range(N)]) # final projection

    self.ffn1 = nn.ModuleList([nn.Linear(d_model, 4 * d_model) for _ in range(N)]) # ffn layer
    self.ffn2 = nn.ModuleList([nn.Linear(4 * d_model, d_model) for _ in range(N)])

    self.ln1 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(N)])
    self.ln2 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(N)])

    self.dropout = nn.Dropout(0.1)

    self.fc = nn.Linear(d_model, d_i // N_p) # final layer

  def multi_head_attention(self, x, layer_idx):
    W_tot = []
    for Q, K, V in zip(self.W_q[layer_idx], self.W_k[layer_idx], self.W_v[layer_idx]):
      Q_i = Q(x)
      K_i = K(x)
      V_i = V(x)

      alignment = torch.matmul(Q_i, K_i.transpose(-2, -1))                      # query-key alignment

      wei = self.dropout(torch.softmax(alignment / (d_k ** 0.5), dim=-1))       # alignment weights
      wei_value = torch.matmul(wei, V_i)                                        # weighted values

      W_tot.append(wei_value)

    out = self.W_o[layer_idx](torch.cat(W_tot, dim=2))
    return out

  def forward(self, x): # (B, N, P^2 * C)
    p = torch.arange(x.size(1)).to(x.device)
    x = self.dropout(self.embed(x) + self.pos(p))     # word & positional embedding
    for i in range(N):
      out = self.multi_head_attention(x, i)  # multi head attention
      out = self.ln1[i](out + x)             # layernorm + residual connection

      fn = self.ffn1[i](out)                 # ffn
      fn = torch.relu(fn)
      fn = self.dropout(self.ffn2[i](fn))

      out = self.ln2[i](fn + out)
      x = out

    out = self.fc(out).flatten(start_dim=1)
    return out


In [34]:
class miniCLIP(nn.Module):
  def __init__(self):
    super().__init__()

    self.transformer = Transformer()
    self.vision_transformer = VisionTransformer()

    self.W_i = nn.Linear(d_i, d_e)
    self.W_t = nn.Linear(d_t, d_e)

  def forward(self, img, caption):
    I_f = self.vision_transformer(img) # n x d_i
    T_f = self.transformer(caption)    # n x d_t

    I_e = F.normalize(self.W_i(I_f), p=2, dim=1)
    T_e = F.normalize(self.W_t(T_f), p=2, dim=1)

    logits = torch.matmul(I_e, T_e.T)
    return logits


In [35]:
model = miniCLIP().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

for epoch in range(epochs):
    total_loss = 0.0
    pbar = tqdm(dataloader, desc=f"Epoch {epoch + 1}/{epochs}")
    model.train()

    for img, caption in pbar:
        optimizer.zero_grad()
        img, caption = img.to(device), caption.to(device)
        labels = torch.arange(img.size(0)).to(device)

        out = model(img, caption)
        loss_i = criterion(out, labels)
        loss_t = criterion(out.T, labels)
        loss = (loss_i + loss_t) / 2

        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        img_batch, cap_batch = next(iter(dataloader))
        img_batch, cap_batch = img_batch.to(device), cap_batch.to(device)
        logits = model(img_batch, cap_batch)
        diag = logits.diag().mean().item()
        off_diag = (logits.sum() - logits.diag().sum()) / (batch_size * (batch_size - 1))

    print(f"Epoch: {epoch + 1}/{epochs} | Loss: {total_loss / len(dataloader):.4f} | Gap: {diag - off_diag.item():.4f}")

Epoch 1/10:   9%|▉         | 229/2484 [01:47<17:37,  2.13it/s, loss=4.1589]


KeyboardInterrupt: 